# Quality Drift Monitor

Pulls historical eval scores from Langfuse, plots metric trends, and flags regressions.

Set `USE_SYNTHETIC_DATA = True` on fresh installs (< 14 days of real scores).

In [ ]:
USE_SYNTHETIC_DATA = True  # Set False once 14+ days of real scores exist
HISTORY_DAYS = 30
WINDOW_DAYS = 7

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Fetch scores

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone

if USE_SYNTHETIC_DATA:
    # Generate 30 days of synthetic scores with a realistic drift pattern
    rng = np.random.default_rng(42)
    now = datetime.now(timezone.utc)
    metrics = {
        'faithfulness': (0.82, -0.003),     # slight downward drift
        'answer_relevancy': (0.78, 0.001),  # stable
        'contextual_relevancy': (0.75, -0.004),  # moderate drift
        'hallucination': (0.08, 0.005),     # slight increase (worse)
    }
    rows = []
    for day in range(HISTORY_DAYS, 0, -1):
        ts = now - timedelta(days=day)
        n = rng.integers(3, 8)
        for metric, (base, drift_per_day) in metrics.items():
            trend = base + drift_per_day * (HISTORY_DAYS - day)
            values = rng.normal(trend, 0.03, n).clip(0, 1)
            for v in values:
                rows.append({'timestamp': ts, 'trace_id': f'synthetic-{day}', 'metric': metric, 'value': float(v), 'model': 'synthetic'})
    df_scores = pd.DataFrame(rows)
    print(f'Synthetic: {len(df_scores)} score rows across {HISTORY_DAYS} days')
else:
    from app.eval.drift import fetch_scores_from_langfuse
    df_scores = fetch_scores_from_langfuse(days=HISTORY_DAYS)
    print(f'Langfuse: {len(df_scores)} score rows')

df_scores.head()

## 2. Trend plots

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

metric_list = df_scores['metric'].unique().tolist()
fig = make_subplots(rows=len(metric_list), cols=1, shared_xaxes=True,
                    subplot_titles=metric_list, vertical_spacing=0.06)

for i, metric in enumerate(metric_list, 1):
    mdf = df_scores[df_scores['metric'] == metric].copy()
    mdf['date'] = pd.to_datetime(mdf['timestamp']).dt.date
    daily = mdf.groupby('date')['value'].mean().reset_index()
    daily['rolling7'] = daily['value'].rolling(7, min_periods=1).mean()

    fig.add_trace(go.Scatter(x=daily['date'], y=daily['value'], mode='markers',
                             name=metric, marker=dict(size=4, opacity=0.5),
                             showlegend=False), row=i, col=1)
    fig.add_trace(go.Scatter(x=daily['date'], y=daily['rolling7'], mode='lines',
                             name=f'{metric} (7d avg)', line=dict(width=2)),
                  row=i, col=1)

fig.update_layout(height=250 * len(metric_list), title_text='Metric Trends (7-day rolling mean)',
                  template='plotly_white')
fig.show()

## 3. Regression detection

In [ ]:
from app.eval.drift import check_drift, print_drift_table

alerts = check_drift(df_scores, window_days=WINDOW_DAYS)
print_drift_table(alerts, all_metrics=metric_list)

## 4. Programmatic check (importable from CLI or CI)

In [ ]:
# check_drift() can be imported and called from app/cli/commands/drift.py
# Usage: python -m app.main drift-check --fail-on-regression

if alerts:
    print(f'⚠  {len(alerts)} regression(s) detected: {[a.metric for a in alerts]}')
else:
    print('✅ No regressions detected.')